### Import Library

In [1]:
import requests
import json

### Fetch Data From API

In [2]:
def get_response(pageIndex=1):
  url = "https://api.gms.moontontech.com/api/gms/source/2669606/2756564"
  params = {
    "pageSize":21,"pageIndex":pageIndex,"filters":[
      {"field":"<hero.data.sortid>","operator":"hasAnyOf","value":[1,2,3,4,5,6]},
      {"field":"<hero.data.roadsort>","operator":"hasAnyOf","value":[1,2,3,4,5]}
    ],
    "sorts":[{"data":{"field":"hero_id","order":"desc"},"type":"sequence"}],
    "fields":["id","hero_id","hero.data.name","hero.data.smallmap","hero.data.sortid","hero.data.roadsort"],
    "object":[]
  }

  response = requests.post(url, json=params)
  if response.status_code == 200:
    return response.json()
  else:
    raise Exception("Error code:", response.status_code)

In [3]:
heroes = {}

for i in range(1, 8):
  response = get_response(i)
  if response:
    data = response['data']['records']
    for d in data:
      name = d['data']['hero']['data']['name'].lower()
      heroes[name] = d['data']['hero_id'] - 1

### Save Heroes to JSON

In [5]:
# heroes2id
with open('../data/heroes2id.json', 'w', encoding='utf-8') as file:
  json.dump(heroes, file, indent=2)

# swap keys and values
id2heroes = {v: k for k, v in heroes.items()}
with open('../data/id2heroes.json', 'w', encoding='utf-8') as file:
  json.dump(id2heroes, file, indent=2)

### Download Hero Images

In [33]:
def download_head_hero(id_hero):
  url = "https://api.gms.moontontech.com/api/gms/source/2669606/2756564"
  params = {
    "pageSize":20,"pageIndex":1,"filters":[
      {"field":"hero_id","operator":"eq","value":id_hero}
      ],"sorts":[],"object":[]
    }
  response = requests.post(url, json=params)
  data = response.json()['data']['records'][0]['data']['hero']['data']
  image = requests.get(data['head']).content
  with open(f'../hero_images/{data["name"].lower()}.png', 'wb') as file:
    file.write(image)

In [34]:
for i in range(1, 129):
  download_head_hero(i)